# How Markets Work?

Before you can build a strategy, you need to understand the machine your orders flow into. A market is not a magic place where "the price" lives — it is a network of exchanges, brokers, and other traders, all interacting through a strict set of rules. This lesson takes you behind the curtain so that when you later model fills, costs, and liquidity, you know exactly what is happening to your order.

By the end of this lesson you will be able to:

- Name the main participants in a market and what each one does
- Explain step by step how a trade actually gets matched
- Distinguish primary from secondary markets
- Define "price" precisely as last trade versus bid and ask
- Describe market hours, opening and closing auctions, and the role of liquidity providers



## 1. Who is in the market?

A market is just a structured meeting place for people who want to buy and people who want to sell. The key players are:

- **Exchanges**. A central venue — like Nasdaq, the NYSE, the CME, or Binance — that maintains the official record of orders and matches buyers with sellers under published rules. The exchange does not take a view on price; it is a referee and a matchmaker.
- **Brokers**. You almost never connect to an exchange directly. A broker (Interactive Brokers, Alpaca, a crypto exchange's retail interface) is the licensed intermediary that holds your account, routes your orders, and handles settlement. As a quant, your code talks to the broker's API, and the broker talks to the exchange.
- **Buyers and sellers**. Everyone with an order: pension funds, hedge funds, banks, and individuals like you. At any instant the market is the sum of everyone's intentions to trade.
- **Market makers / liquidity providers**. Specialised firms that continuously post both a price to buy and a price to sell, profiting from the small gap between them. They are the reason *you can usually trade right now instead of waiting for a natural counterparty to appear*.

>A useful mental model: the exchange holds a list of everyone's resting offers, and a trade happens the instant a buyer's willingness to pay meets a seller's willingness to accept.

It's worth understanding the chain of intermediaries your order actually traverses, because each link costs time and sometimes money. When your code submits an order, it goes to your **broker**, who may route it to an exchange, to a different venue, or to a **wholesaler** (a firm that pays the broker for the right to fill retail orders — the practice behind "payment for order flow"). Only after all that routing does your order interact with the actual book. Each hop adds latency and, potentially, a small worsening of your price. You don't need to micromanage routing as a beginner, but you should know it exists, because it explains why your fills sometimes differ slightly from what the screen implied.



## 2. How a trade gets matched?

Most modern markets run a **central limit order book** with two simple priority rules: 
- best price first, and 
- among equal prices, earliest order first (price-time priority).

Walk through a concrete sequence:

1. A seller posts "I'll sell 100 shares at $50.05." This is a resting **limit order** — it sits in the book, waiting.
2. Another seller posts "I'll sell 200 shares at $50.06." It rests behind the first, at a worse price for buyers.
3. You send a market order to buy 100 shares — "buy at whatever the best available price is."
4. The exchange's matching engine pairs your buy with the cheapest seller: you get 100 shares at $50.05. A trade is printed.
5. That $50.05 offer is now gone. The next buyer to come in would trade against the $50.06 offer.

The matching engine does this millions of times a second, deterministically. There is no human deciding who trades; it is pure rule-following on a sorted list of orders.

The two priority rules deserve a moment's reflection because they shape strategy design.
- **Price priority** means the market always serves the best-priced order first -a buyer willing to pay more gets filled before one bidding lower.
- **Time priority** breaks ties: among orders at the same price, the one that arrived earliest fills first.

This is why, in fast strategies, being early in the queue at a given price level is itself valuable — your order ahead of 10,000 shares at the same price won't trade until those 10,000 do. We'll return to this "queue position" idea when we discuss maker orders and order types.


## 3. Primary vs Secondary Markets

These two words confuse many beginners, so pin them down:

1. **Primary market**. Where a security is created and sold for the first time. When a company does an IPO, or a government issues new bonds, the money goes to the issuer. This is a one-time event for each batch of securities.
2. **Secondary market**. Where already-issued securities are traded between investors. The company gets nothing when you buy its shares on the exchange — you are buying from another investor. This is where essentially all of your trading as a quant happens.

The secondary market is what gives the primary market value: nobody would buy a new issue if they could never sell it later. Liquidity in the secondary market is what you depend on every day.

A concrete way to see the distinction: when a company IPOs at $20 per share, that $20 (times the number of new shares) flows into the company's bank account to fund its business — that's the primary market doing its job. The very next second, when those shares start trading on the exchange and the price moves to $22, no money reaches the company at all; one investor simply paid another investor $22. Every trade you will ever place as a quant is of this second kind. You are a participant in the secondary market, trading claims that already exist, and your profits and losses come entirely from other investors, never from the issuing companies.

## 4. What "Price" really is?

Beginners say "the price of the stock is $50." But there is no single price — there are several numbers, and confusing them causes real trading mistakes.

1. **Bid**. The highest price someone is currently willing to pay. This is the price you can sell at right now.
2. **Ask (or offer)**. The lowest price someone is currently willing to sell at. This is the price you can buy at right now.
3. **Spread**. The gap between bid and ask. It is a real cost you pay to trade immediately.
4. **Last trade**. The price at which the most recent transaction actually happened. This is the number quoted on most apps and charts — but it is history, not a price you can necessarily get.

Here is the distinction in a tiny quote snapshot:



If you send a market buy, you pay 50.06, not the 50.05 "last" price on your screen. That two-cent difference, multiplied across thousands of trades, is exactly the kind of cost a serious backtest must include. We devote a full lesson to spread and slippage shortly.

It helps to commit a single sentence to memory: **you buy at the ask and sell at the bid**. From the market's point of view, you always trade at the price that is worse for you — you cross the spread to get done. The "mid price," halfway between bid and ask, is a useful fiction for measuring fair value and computing returns, but you cannot generally trade there with a simple order; it's where the two sides would meet if they compromised, not a price anyone is actually offering. Beginners who reason as if they trade at the mid systematically overestimate every backtest.

## 5. Market Hours and Auctions

Markets are not open continuously, and the way they open and close matters.

1. **Regular session**. US equities trade roughly 9:30 a.m. to 4:00 p.m. Eastern. This is when liquidity is deepest.
2. **Pre-market and after-hours**. Trading happens outside regular hours but with far fewer participants, wider spreads, and jumpier prices. Risky for naive automated orders.
3. **Opening and closing auctions**. Instead of trading continuously at the open, the exchange collects buy and sell orders for a window, then computes the single price that matches the most volume, and executes everyone at that one price. The closing auction is often the largest, most liquid moment of the day — important if your strategy trades at the close.

Other asset classes differ: futures trade nearly 24 hours on weekdays, and crypto trades 24/7 with no auctions and no official close. Knowing your instrument's calendar is part of knowing your strategy.

The auction mechanism is more clever than it first appears, and worth understanding because many systematic strategies trade specifically at the close. Rather than matching orders one at a time, an auction gathers all the buy and sell interest for a stock over a short window and then solves a single optimization: find the one price at which the largest number of shares can trade.Everyone who participates is filled at that same uniform price, regardless of what limit they entered. This concentration of liquidity into a single moment is why the closing auction is the deepest, lowest-impact time to trade large size in equities — and why index funds, which must trade at the official close, do so much of their business there. If your strategy generates signals on the close, the closing auction is often exactly where you want to execute.

## 6. The Role of Liquidity Providers

Why is there almost always someone to trade with? Because market makers are paid to be there. A market maker might continuously quote:


If a seller hits their bid (they buy at 50.04) and shortly after a buyer lifts their ask (they sell at 50.06), the market maker pockets the two-cent spread for providing immediacy. Exchanges often give them rebates to encourage this. The result for you: tighter spreads and the ability to trade on demand. When market makers step away — during news shocks or thin hours — spreads blow out and your costs spike. Much of practical execution is about trading when *liquidity is present*.

>Question to Ponder: During times of massive downside or upside . How does market maker react?. Does Volume show the sign?

It's important to see that the spread is the market maker's compensation for risk, not a fee they collect for free. Between buying at the bid and selling at the ask, the market maker holds inventory, and the price can move against them — if bad news hits right after they buy, they're stuck with a losing position.They widen their quotes precisely when this risk is high (around news, in thin hours, during volatility), which is why spreads blow out exactly when you most want to trade. Understanding this turns "the spread" from an arbitrary toll into something predictable: it is the price of immediacy, and that price rises with uncertainty.

## 6. Worked example: walking through one order
Suppose the book for a stock looks like this:

You send a market order to buy *300* shares. The engine fills you against the cheapest asks first:

1. *200* shares at 50.06
2. *100* shares at 50.07

Your average fill price is (`200*50.06 + 100*50.07) / 300 = 50.0633`. You wanted "the price," around 50.06, but buying more than the top level pushed you up to 50.07 for the rest. That upward drift is **market impact**, and it is the direct consequence of how matching works. The bigger your order relative to the book, the worse your average price.

Let's verify and extend that with code, and see what happens if your order is even larger relative to the book:

In [1]:
# (price, size), cheapest first
asks = [(50.60,200),(50.70,150),(50.80,100)] 

def market_buy(ask, qty):
    remaining = qty
    cost = 0.0
    filled = 0
    for price, size in asks:
        take = min(remaining,size)
        cost += take * price
        filled += take
        remaining -= take
        if remaining == 0:
            break
    avg = cost / filled if filled else None
    return filled, avg, remaining

for order in (300,450,600):
    filled, avg, remaining = market_buy(asks, order)
    print(f"order {order}: filled {filled} @ avg {avg:.4f}, unfilled {remaining}")




order 300: filled 300 @ avg 50.6333, unfilled 0
order 450: filled 450 @ avg 50.6778, unfilled 0
order 600: filled 450 @ avg 50.6778, unfilled 150


The 300-share order fills at 50.0633, as computed by hand. The 450-share order reaches into the third level and fills at a worse average. The 600-share order can't fully fill — the book only holds 450 shares across these three levels, so 150 shares are left unexecuted, and in reality your order would either rest waiting for new sellers or push the price up further. This is the single most important intuition about real markets: **liquidity is finite**, and demanding more of it than the book holds moves the price against you and can leave you partially unfilled.

## 7. Common Mistakes
- Treating "last price" as the price you can trade at. You buy at the ask and sell at the bid; last trade is just a historical print.
- Ignoring the spread as a cost. Every round trip pays it at least once. For fast strategies this dominates everything.
- Trading in thin hours by accident. Pre-market and the first minute after open have wide spreads and erratic fills; naive market orders there get badly filled.
- Assuming infinite liquidity. A backtest that fills your whole order at one price is lying to you the moment your size approaches the book's depth.
- Forgetting the venue's calendar. Sending equity orders at 3 a.m. or expecting a crypto "close" leads to broken logic.



## 8. Key Takeways

1. A market is a rule-bound network of exchanges, brokers, traders, and market makers, not a single place where one price lives.
2. Trades match by price-time priority in a central limit order book, automatically and deterministically.
3. Primary markets create securities; secondary markets trade them between investors — and that's where you operate.
4. "Price" is really the bid, ask, spread, and last trade — you buy at the ask and sell at the bid, never the mid.
5. Liquidity providers make on-demand trading possible and the spread is their pay for risk; when they retreat, your costs explode, so when you trade matters as much as what.